# Curva ABC de Produtos

## Objetivo

Este notebook classifica os produtos da **Distribuidora Horizonte** utilizando a metodologia Curva ABC.

A classificação considera a participação de cada produto no faturamento dos últimos 12 meses disponíveis na base.

## Classes

- **Classe A:** produtos responsáveis pelos primeiros 80% do faturamento acumulado;
- **Classe B:** produtos responsáveis pela faixa entre 80% e 95% do faturamento acumulado;
- **Classe C:** produtos responsáveis pelo restante do faturamento.

## Indicadores analisados

- faturamento;
- lucro bruto;
- margem;
- quantidade vendida;
- pedidos;
- clientes compradores;
- participação no faturamento;
- participação acumulada;
- classificação ABC.

Além da classificação comercial, os resultados serão posteriormente combinados com a posição de estoque para identificar situações como:

- produto Classe A com risco de ruptura;
- produto Classe A abaixo do estoque mínimo;
- produto Classe C com excesso de estoque.

## Saídas

Este notebook cria:

- `produtos_curva_abc`;
- `resumo_curva_abc`.

## Fluxo

Silver → Vendas dos últimos 12 meses → Curva ABC → Gold

## Resultado

A Curva ABC dos produtos foi construída utilizando o faturamento dos últimos 12 meses disponíveis.

Os produtos foram classificados de acordo com sua participação acumulada no faturamento:

- Classe A: maior relevância financeira;
- Classe B: relevância intermediária;
- Classe C: menor participação no faturamento;
- Sem venda: produtos sem movimentação no período analisado.

A análise também disponibiliza indicadores de:

- faturamento;
- lucro;
- margem;
- quantidade vendida;
- clientes compradores;
- pedidos;
- venda média diária.

A venda média diária será utilizada na próxima etapa para calcular a cobertura de estoque e apoiar decisões de reposição.

A classificação ABC isoladamente não determina a necessidade de compra. Na próxima análise, a importância comercial do produto será combinada com sua disponibilidade em estoque.

In [0]:
from datetime import timedelta

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"
schema_gold = "varejo_gold"


print(f"Catálogo: {catalogo_atual}")
print(f"Origem: {schema_silver}")
print(f"Destino: {schema_gold}")

In [0]:
def carregar_silver(nome_tabela):
    """
    Carrega uma tabela da camada Silver.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


def salvar_gold(
    dataframe,
    nome_tabela
):
    """
    Salva um DataFrame como tabela Delta
    gerenciada na camada Gold.
    """

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )

    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )

    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
df_resultado_qualidade = spark.table(
    f"{catalogo_atual}."
    f"{schema_silver}."
    f"resultado_qualidade"
)


testes_criticos_reprovados = (

    df_resultado_qualidade

    .filter(
        (F.col("status") == "REPROVADO")
        &
        (F.col("criticidade") == "CRÍTICA")
    )

    .count()
)


if testes_criticos_reprovados > 0:

    raise Exception(
        "Existem testes críticos de qualidade "
        "reprovados. A Curva ABC não será processada."
    )


print(
    "Quality Gate aprovado."
)

In [0]:
df_vendas = carregar_silver(
    "fato_vendas"
)


df_produtos = carregar_silver(
    "dim_produto"
)

In [0]:
ultima_data_venda = (

    df_vendas

    .agg(
        F.max(
            "data_venda"
        ).alias(
            "ultima_data_venda"
        )
    )

    .first()[
        "ultima_data_venda"
    ]
)


print(
    f"Última data disponível: "
    f"{ultima_data_venda}"
)

In [0]:
df_periodo_referencia = (

    spark.createDataFrame(
        [(ultima_data_venda,)],
        ["data_fim"]
    )

    .withColumn(
        "data_inicio",
        F.add_months(
            F.col("data_fim"),
            -12
        )
    )
)


periodo = (
    df_periodo_referencia
    .first()
)


data_inicio = periodo[
    "data_inicio"
]

data_fim = periodo[
    "data_fim"
]


print(
    f"Período da Curva ABC: "
    f"{data_inicio} até {data_fim}"
)

In [0]:
df_vendas_12m = (

    df_vendas

    .filter(
        (F.col("data_venda") >= F.lit(data_inicio))
        &
        (F.col("data_venda") <= F.lit(data_fim))
    )
)


print(
    f"Itens considerados: "
    f"{df_vendas_12m.count():,}"
)

In [0]:
df_desempenho_produto = (

    df_vendas_12m

    .groupBy(
        "id_produto"
    )

    .agg(

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.round(
            F.sum(
                "custo_total"
            ),
            2
        ).alias(
            "custo_total"
        ),

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_vendida"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "quantidade_pedidos"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "clientes_compradores"
        ),

        F.min(
            "data_venda"
        ).alias(
            "primeira_venda_periodo"
        ),

        F.max(
            "data_venda"
        ).alias(
            "ultima_venda_periodo"
        )
    )
)

In [0]:
df_desempenho_produto = (

    df_produtos.alias("p")

    .join(
        df_desempenho_produto.alias("v"),
        on="id_produto",
        how="left"
    )

    .select(

        "id_produto",

        F.col(
            "p.nome_produto"
        ).alias(
            "nome_produto"
        ),

        F.col(
            "p.categoria"
        ).alias(
            "categoria"
        ),

        F.col(
            "p.subcategoria"
        ).alias(
            "subcategoria"
        ),

        F.col(
            "p.marca"
        ).alias(
            "marca"
        ),

        F.col(
            "p.situacao_produto"
        ).alias(
            "situacao_produto"
        ),

        F.coalesce(
            F.col("v.faturamento"),
            F.lit(0)
        ).alias(
            "faturamento"
        ),

        F.coalesce(
            F.col("v.lucro_bruto"),
            F.lit(0)
        ).alias(
            "lucro_bruto"
        ),

        F.coalesce(
            F.col("v.custo_total"),
            F.lit(0)
        ).alias(
            "custo_total"
        ),

        F.coalesce(
            F.col(
                "v.quantidade_vendida"
            ),
            F.lit(0)
        ).alias(
            "quantidade_vendida"
        ),

        F.coalesce(
            F.col(
                "v.quantidade_pedidos"
            ),
            F.lit(0)
        ).alias(
            "quantidade_pedidos"
        ),

        F.coalesce(
            F.col(
                "v.clientes_compradores"
            ),
            F.lit(0)
        ).alias(
            "clientes_compradores"
        ),

        F.col(
            "v.primeira_venda_periodo"
        ),

        F.col(
            "v.ultima_venda_periodo"
        )
    )
)

In [0]:
df_desempenho_produto = (

    df_desempenho_produto

    .withColumn(

        "margem_percentual",

        F.when(
            F.col("faturamento") > 0,

            F.round(
                F.col("lucro_bruto")
                /
                F.col("faturamento")
                * 100,
                2
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
faturamento_total = (

    df_desempenho_produto

    .agg(
        F.sum(
            "faturamento"
        ).alias(
            "faturamento_total"
        )
    )

    .first()[
        "faturamento_total"
    ]
)


print(
    f"Faturamento analisado: "
    f"R$ {faturamento_total:,.2f}"
)

In [0]:
df_curva_abc = (

    df_desempenho_produto

    .withColumn(

        "participacao_faturamento_percentual",

        F.when(
            F.lit(faturamento_total) > 0,

            F.round(
                F.col("faturamento")
                /
                F.lit(faturamento_total)
                * 100,
                4
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
janela_ordenacao = (

    Window

    .orderBy(
        F.desc(
            "faturamento"
        ),
        F.asc(
            "id_produto"
        )
    )
)

In [0]:
df_curva_abc = (

    df_curva_abc

    .withColumn(

        "ranking_faturamento",

        F.row_number()
        .over(
            janela_ordenacao
        )
    )
)

In [0]:
janela_acumulada = (

    Window

    .orderBy(
        F.desc(
            "faturamento"
        ),
        F.asc(
            "id_produto"
        )
    )

    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

In [0]:
df_curva_abc = (

    df_curva_abc

    .withColumn(

        "faturamento_acumulado",

        F.sum(
            "faturamento"
        )
        .over(
            janela_acumulada
        )
    )

    .withColumn(

        "participacao_acumulada_percentual",

        F.round(
            F.col(
                "faturamento_acumulado"
            )
            /
            F.lit(
                faturamento_total
            )
            * 100,
            4
        )
    )
)

In [0]:
df_curva_abc = (

    df_curva_abc

    .withColumn(

        "classe_abc",

        F.when(
            F.col(
                "participacao_acumulada_percentual"
            ) <= 80,

            F.lit("A")
        )

        .when(
            F.col(
                "participacao_acumulada_percentual"
            ) <= 95,

            F.lit("B")
        )

        .otherwise(
            F.lit("C")
        )
    )
)

In [0]:
df_curva_abc = (

    df_curva_abc

    .withColumn(

        "classe_abc",

        F.when(
            F.col(
                "faturamento"
            ) == 0,

            F.lit(
                "Sem venda"
            )
        )

        .otherwise(
            F.col(
                "classe_abc"
            )
        )
    )
)

In [0]:
dias_periodo = (
    data_fim - data_inicio
).days + 1


print(
    f"Dias analisados: {dias_periodo}"
)

In [0]:
df_curva_abc = (

    df_curva_abc

    .withColumn(

        "venda_media_diaria",

        F.round(
            F.col(
                "quantidade_vendida"
            )
            /
            F.lit(
                dias_periodo
            ),
            4
        )
    )
)

In [0]:
df_curva_abc = (

    df_curva_abc

    .withColumn(
        "data_inicio_analise",
        F.lit(
            data_inicio
        )
    )

    .withColumn(
        "data_fim_analise",
        F.lit(
            data_fim
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_produtos_curva_abc = (

    df_curva_abc

    .select(

        "id_produto",
        "nome_produto",

        "categoria",
        "subcategoria",
        "marca",
        "situacao_produto",

        "ranking_faturamento",

        "faturamento",
        "lucro_bruto",
        "margem_percentual",

        "quantidade_vendida",
        "venda_media_diaria",

        "quantidade_pedidos",
        "clientes_compradores",

        "participacao_faturamento_percentual",
        "faturamento_acumulado",
        "participacao_acumulada_percentual",

        "classe_abc",

        "primeira_venda_periodo",
        "ultima_venda_periodo",

        "data_inicio_analise",
        "data_fim_analise",

        "_data_processamento"
    )
)

In [0]:
salvar_gold(
    df_produtos_curva_abc,
    "produtos_curva_abc"
)

In [0]:
display(

    df_produtos_curva_abc

    .select(
        "ranking_faturamento",
        "id_produto",
        "nome_produto",
        "categoria",
        "faturamento",
        "participacao_faturamento_percentual",
        "participacao_acumulada_percentual",
        "classe_abc"
    )

    .orderBy(
        "ranking_faturamento"
    )

    .limit(50)
)

In [0]:
display(

    df_produtos_curva_abc

    .groupBy(
        "classe_abc"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_produtos"
        ),

        F.round(
            F.sum(
                "faturamento"
            ),
            2
        ).alias(
            "faturamento"
        )
    )

    .orderBy(
        F.desc(
            "faturamento"
        )
    )
)

In [0]:
total_produtos = (
    df_produtos_curva_abc.count()
)

In [0]:
df_resumo_curva_abc = (

    df_produtos_curva_abc

    .groupBy(
        "classe_abc"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_produtos"
        ),

        F.round(
            F.sum(
                "faturamento"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.sum(
            "quantidade_vendida"
        ).alias(
            "quantidade_vendida"
        ),

        F.round(
            F.avg(
                "margem_percentual"
            ),
            2
        ).alias(
            "margem_media_percentual"
        )
    )
)

In [0]:
df_resumo_curva_abc = (

    df_resumo_curva_abc

    .withColumn(

        "participacao_produtos_percentual",

        F.round(
            F.col(
                "quantidade_produtos"
            )
            /
            F.lit(
                total_produtos
            )
            * 100,
            2
        )
    )

    .withColumn(

        "participacao_faturamento_percentual",

        F.when(
            F.lit(
                faturamento_total
            ) > 0,

            F.round(
                F.col(
                    "faturamento"
                )
                /
                F.lit(
                    faturamento_total
                )
                * 100,
                2
            )
        )

        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(
        "data_inicio_analise",
        F.lit(
            data_inicio
        )
    )

    .withColumn(
        "data_fim_analise",
        F.lit(
            data_fim
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_resumo_curva_abc,
    "resumo_curva_abc"
)

In [0]:
display(

    df_resumo_curva_abc

    .select(

        "classe_abc",

        "quantidade_produtos",
        "participacao_produtos_percentual",

        "faturamento",
        "participacao_faturamento_percentual",

        "lucro_bruto",
        "margem_media_percentual"
    )

    .orderBy(
        F.desc(
            "faturamento"
        )
    )
)

In [0]:
display(

    df_produtos_curva_abc

    .filter(
        F.col(
            "classe_abc"
        ) == "A"
    )

    .select(
        "ranking_faturamento",
        "nome_produto",
        "categoria",
        "faturamento",
        "lucro_bruto",
        "margem_percentual",
        "quantidade_vendida",
        "clientes_compradores"
    )

    .orderBy(
        "ranking_faturamento"
    )

    .limit(20)
)

In [0]:
display(

    df_produtos_curva_abc

    .filter(
        F.col(
            "classe_abc"
        ) == "A"
    )

    .select(
        "nome_produto",
        "categoria",
        "faturamento",
        "lucro_bruto",
        "margem_percentual"
    )

    .orderBy(
        F.asc(
            "margem_percentual"
        )
    )

    .limit(20)
)

In [0]:
display(

    df_produtos_curva_abc

    .filter(
        F.col(
            "classe_abc"
        ) == "Sem venda"
    )

    .select(
        "id_produto",
        "nome_produto",
        "categoria",
        "situacao_produto"
    )
)

In [0]:
faturamento_silver_periodo = (

    df_vendas_12m

    .agg(
        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        )
    )

    .first()[
        "faturamento"
    ]
)


faturamento_abc = (

    df_produtos_curva_abc

    .agg(
        F.round(
            F.sum(
                "faturamento"
            ),
            2
        ).alias(
            "faturamento"
        )
    )

    .first()[
        "faturamento"
    ]
)


print(
    f"Silver: R$ "
    f"{faturamento_silver_periodo:,.2f}"
)

print(
    f"ABC:    R$ "
    f"{faturamento_abc:,.2f}"
)

In [0]:
diferenca = abs(
    float(
        faturamento_silver_periodo
    )
    -
    float(
        faturamento_abc
    )
)


if diferenca > 0.05:

    raise Exception(
        "O faturamento da Curva ABC "
        "não corresponde ao faturamento "
        "da Silver no período."
    )


print(
    "Validação concluída: "
    "faturamento da Curva ABC consistente."
)

In [0]:
spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`produtos_curva_abc`
    IS 'Classificação dos produtos por Curva ABC utilizando o faturamento acumulado dos últimos 12 meses.'
    """
)


spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`resumo_curva_abc`
    IS 'Indicadores consolidados das classes A, B e C dos produtos.'
    """
)


print(
    "Descrições adicionadas às tabelas da Curva ABC."
)